In [1]:
import pandas as pd
import numpy as np

In [2]:
production = pd.read_csv(
    "../datasets/cleaned/crop_production_clean.csv"
)

weather = pd.read_parquet(
    "../datasets/processed/state_monthly_weather.parquet"
)

In [3]:
print(production.shape)
print(weather.shape)

(242361, 8)
(7980, 9)


In [4]:
production.head()

,State_Name,District_Name,Crop_Year,Season,Crop,Area,Production,Yield
0,Andaman And Nicobar Islands,Nicobars,2000,Kharif,Arecanut,1254.0,2000.0,1.594896
1,Andaman And Nicobar Islands,Nicobars,2000,Kharif,Other Kharif Pulses,2.0,1.0,0.500000
2,Andaman And Nicobar Islands,Nicobars,2000,Kharif,Rice,102.0,321.0,3.147059
3,Andaman And Nicobar Islands,Nicobars,2000,Whole Year,Banana,176.0,641.0,3.642045
4,Andaman And Nicobar Islands,Nicobars,2000,Whole Year,Cashewnut,720.0,165.0,0.229167


In [5]:
weather.head()

,State_Name,Year,Month,Avg_Temperature_C,Avg_DewPoint_C,Avg_Rainfall_mm,Avg_Wind_Speed,Avg_Surface_Pressure_hPa,Grid_Points
0,Andaman And Nicobar Islands,1997,1,25.348749,19.902393,0.292015,4.067673,1010.664429,10
1,Andaman And Nicobar Islands,1997,2,25.340546,19.990643,0.183105,3.886254,1009.292297,10
2,Andaman And Nicobar Islands,1997,3,27.134258,22.181946,0.261116,2.077748,1009.709656,10
3,Andaman And Nicobar Islands,1997,4,27.925085,22.880255,1.249695,1.498398,1008.677246,10
4,Andaman And Nicobar Islands,1997,5,28.313923,24.248028,5.311203,3.772224,1006.205933,10


In [6]:
weather = weather.rename(
    columns={
        "State_Name": "State"
    }
)

In [7]:
weather_mapping = {
    "Andaman & Nicobar": "Andaman And Nicobar Islands",
    "Chhattishgarh": "Chhattisgarh",
    "Tamilnadu": "Tamil Nadu",
    "Telengana": "Telangana",
    "Jammu and Kashmir": "Jammu And Kashmir"
}

weather["State"] = weather["State"].replace(weather_mapping)

In [8]:
print(sorted(set(production["State_Name"]) - set(weather["State"])))

[]


In [9]:
print(sorted(set(weather["State"]) - set(production["State_Name"])))

['Delhi', 'Ladakh']


In [10]:
yearly_weather = (
    weather
    .groupby(["State", "Year"])
    .agg(
        Avg_Annual_Temperature=("Avg_Temperature_C", "mean"),
        Avg_Annual_DewPoint=("Avg_DewPoint_C", "mean"),
        Total_Annual_Rainfall=("Avg_Rainfall_mm", "sum"),
        Avg_Annual_Wind_Speed=("Avg_Wind_Speed", "mean"),
        Avg_Annual_Surface_Pressure=("Avg_Surface_Pressure_hPa", "mean"),
        Avg_Grid_Points=("Grid_Points", "mean")
    )
    .reset_index()
)

In [12]:
yearly_weather.shape

(665, 8)

In [13]:
yearly_weather.head()

,State,Year,Avg_Annual_Temperature,Avg_Annual_DewPoint,Total_Annual_Rainfall,Avg_Annual_Wind_Speed,Avg_Annual_Surface_Pressure,Avg_Grid_Points
0,Andaman And Nicobar Islands,1997,26.998451,23.106012,59.192657,4.161923,1008.558105,10.0
1,Andaman And Nicobar Islands,1998,27.389936,23.672470,76.032257,3.819835,1007.596863,10.0
2,Andaman And Nicobar Islands,1999,26.780680,23.286791,81.361771,4.468453,1006.848328,10.0
3,Andaman And Nicobar Islands,2000,26.792810,23.292879,84.931473,4.373926,1006.624207,10.0
4,Andaman And Nicobar Islands,2001,26.887018,23.362188,72.599602,4.496150,1006.968811,10.0


In [14]:
yearly_weather.info()

<class 'pandas.DataFrame'>
RangeIndex: 665 entries, 0 to 664
Data columns (total 8 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   State                        665 non-null    str    
 1   Year                         665 non-null    int32  
 2   Avg_Annual_Temperature       665 non-null    float32
 3   Avg_Annual_DewPoint          665 non-null    float32
 4   Total_Annual_Rainfall        665 non-null    float32
 5   Avg_Annual_Wind_Speed        665 non-null    float32
 6   Avg_Annual_Surface_Pressure  665 non-null    float32
 7   Avg_Grid_Points              665 non-null    float64
dtypes: float32(5), float64(1), int32(1), str(1)
memory usage: 32.7 KB


In [15]:
yearly_weather.isnull().sum()

State                          0
Year                           0
Avg_Annual_Temperature         0
Avg_Annual_DewPoint            0
Total_Annual_Rainfall          0
Avg_Annual_Wind_Speed          0
Avg_Annual_Surface_Pressure    0
Avg_Grid_Points                0
dtype: int64

In [20]:
weather[
    weather["State"] == "Andaman And Nicobar Islands"
][["Year", "Month", "Grid_Points"]].head(15)

,Year,Month,Grid_Points
0,1997,1,10
1,1997,2,10
2,1997,3,10
3,1997,4,10
4,1997,5,10
5,1997,6,10
6,1997,7,10
7,1997,8,10
8,1997,9,10
9,1997,10,10


In [21]:
production = production.rename(
    columns={
        "State_Name": "State",
        "Crop_Year": "Year"
    }
)

In [22]:
production_weather = production.merge(
    yearly_weather,
    on=["State", "Year"],
    how="left"
)

In [23]:
print(production_weather.shape)

production_weather.isnull().sum()

(242361, 14)


State                          0
District_Name                  0
Year                           0
Season                         0
Crop                           0
Area                           0
Production                     0
Yield                          0
Avg_Annual_Temperature         0
Avg_Annual_DewPoint            0
Total_Annual_Rainfall          0
Avg_Annual_Wind_Speed          0
Avg_Annual_Surface_Pressure    0
Avg_Grid_Points                0
dtype: int64

In [24]:
production_weather_train = (
    production_weather[
        production_weather["Year"] <= 2015
    ]
    .copy()
)

In [29]:
print(production_weather_train.shape)

(242361, 14)


In [ ]:
production_weather_train.isnull().sum()

State                          0
District_Name                  0
Year                           0
Season                         0
Crop                           0
Area                           0
Production                     0
Yield                          0
Avg_Annual_Temperature         0
Avg_Annual_DewPoint            0
Total_Annual_Rainfall          0
Avg_Annual_Wind_Speed          0
Avg_Annual_Surface_Pressure    0
Avg_Grid_Points                0
dtype: int64

In [30]:
production_weather_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 242361 entries, 0 to 242360
Data columns (total 14 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   State                        242361 non-null  str    
 1   District_Name                242361 non-null  str    
 2   Year                         242361 non-null  int64  
 3   Season                       242361 non-null  str    
 4   Crop                         242361 non-null  str    
 5   Area                         242361 non-null  float64
 6   Production                   242361 non-null  float64
 7   Yield                        242361 non-null  float64
 8   Avg_Annual_Temperature       242361 non-null  float32
 9   Avg_Annual_DewPoint          242361 non-null  float32
 10  Total_Annual_Rainfall        242361 non-null  float32
 11  Avg_Annual_Wind_Speed        242361 non-null  float32
 12  Avg_Annual_Surface_Pressure  242361 non-null  float32
 13  Avg_Grid_P

In [31]:
production_weather_train.describe()

,Year,Area,Production,Yield,Avg_Annual_Temperature,Avg_Annual_DewPoint,Total_Annual_Rainfall,Avg_Annual_Wind_Speed,Avg_Annual_Surface_Pressure,Avg_Grid_Points
count,242361.000000,2.423610e+05,2.423610e+05,242361.000000,242361.000000,242361.000000,242361.000000,242361.000000,242361.000000,242361.000000
mean,2005.625773,1.216741e+04,5.825034e+05,41.649059,24.695276,17.007135,44.752552,1.479069,963.492737,238.995523
std,4.958285,5.085744e+04,1.706581e+07,817.572839,3.789661,3.620987,23.356813,0.602546,49.418228,135.329941
min,1997.000000,1.000000e-01,0.000000e+00,0.000000,4.529313,-1.021463,6.586140,0.298182,671.264771,1.000000
25%,2002.000000,8.700000e+01,8.800000e+01,0.513514,24.962816,16.106752,31.192074,1.117055,960.100769,121.000000
50%,2006.000000,6.030000e+02,7.290000e+02,1.000000,25.469107,17.424799,38.699268,1.406143,976.734924,216.000000
75%,2010.000000,4.545000e+03,7.023000e+03,2.355450,26.185432,19.176126,50.070232,1.948082,989.380798,343.000000
max,2015.000000,8.580100e+06,1.250800e+09,88000.000000,28.336908,23.765112,175.857422,4.511707,1007.695618,497.000000


In [32]:
production_weather_train.to_parquet(
    "../datasets/modeling/production_dataset_training.parquet",
    index=False
)

production_weather_train.to_csv(
    "../datasets/modeling/production_dataset_training.csv",
    index=False
)